# Notebook 01 — Exploratory Data Analysis

**Purpose**: Understand the raw dataset before any modelling.

Key questions:
1. What is the distribution of the target (`responder_6`)?
2. How many unique dates, times, and symbols are there?
3. What is the missing rate per feature column?
4. Are there obvious temporal trends in the target?
5. How do sample weights vary?

**Run after**: `make prepare-data`  
**Data source**: `data/interim/train.parquet`

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Project imports
import sys; sys.path.insert(0, str(Path('../src')))
from janestreet_forecasting.data import schemas as S
from janestreet_forecasting.paths import INTERIM_DIR

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

In [ ]:
# Load a sample for quick EDA (remove .head() to analyse full dataset)
DATA_PATH = INTERIM_DIR / 'train.parquet'

if not DATA_PATH.exists():
    print(f'Data not found at {DATA_PATH}. Run `make prepare-data` first.')
    print('Using synthetic data for demonstration...')
    from janestreet_forecasting.data.loaders import make_synthetic_dataset
    df = make_synthetic_dataset(n_dates=50, n_times_per_date=50, n_symbols=30)
else:
    df = pl.read_parquet(DATA_PATH)

print(f'Shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Basic Statistics ===')
print(f'Total rows: {len(df):,}')
print(f'Unique date_ids: {df[S.DATE_ID].n_unique():,}')
print(f'Unique time_ids: {df[S.TIME_ID].n_unique():,}')
print(f'Unique symbol_ids: {df[S.SYMBOL_ID].n_unique():,}')
print(f'Date range: {df[S.DATE_ID].min()} → {df[S.DATE_ID].max()}')
print(f'Columns: {len(df.columns)}')

## 2. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

target = df[S.TARGET_COL].drop_nulls().to_numpy()

# Histogram
axes[0].hist(target, bins=100, alpha=0.8, color='steelblue', edgecolor='none')
axes[0].set_title(f'Distribution of {S.TARGET_COL}')
axes[0].set_xlabel(S.TARGET_COL)
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='red', lw=1, ls='--', label='zero')
axes[0].legend()

# Per-date mean target
daily = df.group_by(S.DATE_ID).agg(pl.col(S.TARGET_COL).mean().alias('mean_target'))
daily = daily.sort(S.DATE_ID)
axes[1].plot(daily[S.DATE_ID].to_numpy(), daily['mean_target'].to_numpy(), lw=0.8, alpha=0.9)
axes[1].axhline(0, color='red', lw=1, ls='--')
axes[1].set_title('Daily Mean Target')
axes[1].set_xlabel('date_id')
axes[1].set_ylabel('mean responder_6')

plt.tight_layout()
plt.show()

print(f'Target stats: mean={target.mean():.6f}, std={target.std():.6f}, ')
print(f'  min={target.min():.4f}, max={target.max():.4f}')
print(f'  skew={float(pl.Series(target).skew()):.4f}')

## 3. Missing Value Analysis

In [ ]:
n = len(df)
missing_rates = {
    col: df[col].null_count() / n
    for col in S.FEATURE_COLS
    if col in df.columns
}

missing_df = pl.DataFrame({
    'feature': list(missing_rates.keys()),
    'missing_rate': list(missing_rates.values()),
}).sort('missing_rate', descending=True)

fig, ax = plt.subplots(figsize=(14, 5))
cols = missing_df['feature'].to_list()
vals = missing_df['missing_rate'].to_numpy()
ax.bar(range(len(cols)), vals * 100, color=['red' if v > 0.3 else 'steelblue' for v in vals])
ax.set_xlabel('Feature Index')
ax.set_ylabel('Missing Rate (%)')
ax.set_title('Missing Value Rate per Feature (red = >30%)')
ax.axhline(30, color='red', ls='--', lw=1, label='30% threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Features with >30% missing: {(vals > 0.3).sum()}')
print(f'Features with any missing: {(vals > 0).sum()}')

## 4. Sample Weight Distribution

In [ ]:
weights = df[S.WEIGHT].drop_nulls().to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(weights, bins=50, alpha=0.8, color='orange')
axes[0].set_title('Weight Distribution')
axes[0].set_xlabel('weight')

# Zero-weight rows are common in financial datasets (non-tradeable symbols)
zero_weight_frac = (weights == 0).mean()
axes[1].pie(
    [zero_weight_frac, 1 - zero_weight_frac],
    labels=[f'weight=0 ({zero_weight_frac:.1%})', f'weight>0 ({1-zero_weight_frac:.1%})'],
    autopct='%1.1f%%',
    colors=['lightcoral', 'lightgreen'],
)
axes[1].set_title('Zero vs Non-Zero Weights')
plt.tight_layout()
plt.show()

## 5. Temporal Structure

In [ ]:
# Symbols per date — should be roughly constant
symbols_per_date = df.group_by(S.DATE_ID).agg(
    pl.col(S.SYMBOL_ID).n_unique().alias('n_symbols')
).sort(S.DATE_ID)

# Time steps per date
times_per_date = df.group_by(S.DATE_ID).agg(
    pl.col(S.TIME_ID).n_unique().alias('n_times')
).sort(S.DATE_ID)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(symbols_per_date[S.DATE_ID].to_numpy(), symbols_per_date['n_symbols'].to_numpy())
axes[0].set_title('Unique Symbols per Date')
axes[0].set_xlabel('date_id')

axes[1].plot(times_per_date[S.DATE_ID].to_numpy(), times_per_date['n_times'].to_numpy())
axes[1].set_title('Time Steps per Date')
axes[1].set_xlabel('date_id')
plt.tight_layout()
plt.show()